In [ ]:
%pip install torch torchvision transformers pillow open_clip_torch numpy==1.23

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 2.7 MB/s eta 0:00:000:00:0136m0:00:01:02
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done

In [5]:
%pip install --upgrade pip



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 1.3 MB/s eta 0:00:00m eta 0:00:010:00:010m
  Attempting uninstall: pip
    Found existing installation: pip 24.0
    Uninstalling pip-24.0:
      Successfully uninstalled pip-24.0
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
import numpy as np
from PIL import Image
import open_clip
from transformers import Beit3Processor, Beit3Model
import os

device = "cuda" if torch.cuda.is_available() else "cpu"


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.3.0 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/nhanvo/.pyenv/versions/3.11.13/lib/python3.11/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/nhanvo/.pyenv/versions/3.11.13/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/nhanvo/.pyenv/versions/3.11.13/lib/python3.11/site-packages/ipykernel/kernelapp.py", line 739, in start

In [3]:
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms('hf-hub:apple/DFN5B-CLIP-ViT-H-14', device=device)

clip_model.eval()

def get_clip_embedding(image: Image.Image):
    # image: PIL Image RGB
    with torch.no_grad():
        image_tensor = clip_preprocess(image).unsqueeze(0).to(device)
        emb = clip_model.encode_image(image_tensor)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        return emb.cpu().numpy()[0]  # 1D numpy vector

KeyboardInterrupt: 

In [ ]:
beit3_processor = Beit3Processor.from_pretrained("microsoft/beit-3-base")
beit3_model = Beit3Model.from_pretrained("microsoft/beit-3-base").to(device)
beit3_model.eval()

def get_beit3_image_embedding(image_path_or_img):
    """
    Tạo embedding cho image sử dụng BEiT-3
    """
    # Xử lý nếu input là đường dẫn
    if isinstance(image_path_or_img, str):
        image = Image.open(image_path_or_img).convert("RGB")
    else:
        image = image_path_or_img
        
    with torch.no_grad():
        inputs = beit3_processor(images=image, return_tensors="pt").to(device)
        outputs = beit3_model(**inputs)
        image_features = outputs.image_features
        # Chuẩn hóa vector
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        return image_features.cpu().numpy()[0]

/usr/local/lib/python3.11/dist-packages/transformers/utils/deprecation.py:172: UserWarning: The following named arguments are not valid for `BeitImageProcessor.__init__` and were ignored: 'feature_extractor_type'
  return func(*args, **kwargs)


In [ ]:
%pip install pymilvus